In [1]:
from langchain_core.tools import tool
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

/home/pranav/ml_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
groq_api_key=os.getenv("GROQ_API_KEY")

In [3]:
llm= ChatGroq(model="openai/gpt-oss-120b", api_key="groq_api_key")
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_customer(customer_id: str) -> dict:
    """
    Returns customer information.
    """
    return {
        "id": customer_id,
        "name": "Pranav",
        "balance": 250.0
    }

@tool
def get_orders(customer_id: str) -> list[dict]:
    """
    Returns all orders for a customer.
    """
    return [
        {
            "order_id": "O501",
            "customer_id": customer_id,
            "amount": 300.0,
            "status": "processing"
        },
        {
            "order_id": "O502",
            "customer_id": customer_id,
            "amount": 150.0,
            "status": "shipped"
        }
    ]

@tool
def get_order(order_id: str) -> dict:
    """
    Returns details of a specific order.
    """
    return {
        "order_id": order_id,
        "customer_id": "C101",
        "amount": 300.0,
        "status": "processing",
        "refunded": False
    }

@tool
def cancel_order(order_id: Annotated[str,InjectedToolArg]) -> dict:
    """
    Cancels an order.
    """
    return {
        "order_id": order_id,
        "status": "cancelled"
    }

@tool
def issue_refund(order_id: Annotated[str,InjectedToolArg]) -> dict:
    """
    Issues a refund for an order.
    """
    return {
        "order_id": order_id,
        "refund_amount": 300.0,
        "status": "refunded"
    }

In [4]:
llm_with_tools = llm.bind_tools([
    get_customer,
    get_orders,
    cancel_order,
    get_order,
    issue_refund
])

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage
import json

messages = []  # chat history

while True:

    query = input("You: ")

    if query.lower() == "exit":
        break

    messages.append(HumanMessage(content=query))

    # Keep looping until the LLM gives a final answer
    while True:

        ai_message = llm_with_tools.invoke(messages)

        messages.append(ai_message)

        # No tool calls means the LLM has produced the final answer
        if not ai_message.tool_calls:
            print(ai_message.content)
            break

        # Execute all tool calls requested by the LLM
        for tool_call in ai_message.tool_calls:

            if tool_call["name"] == "get_customer":

                tool_message = get_customer.invoke(tool_call)
                messages.append(tool_message)

                customer = json.loads(tool_message.content)
                customer_id = customer["id"]


            elif tool_call["name"] == "get_orders":

                tool_message = get_orders.invoke(tool_call)
                messages.append(tool_message)

                orders = json.loads(tool_message.content)


            elif tool_call["name"] == "get_order":

                tool_message = get_order.invoke(tool_call)
                messages.append(tool_message)

                order = json.loads(tool_message.content)


            elif tool_call["name"] == "cancel_order":

                # -------------------------
                # BUSINESS RULES
                # -------------------------

                if order["status"] == "shipped":

                    tool_message = ToolMessage(
                        content="Cancellation failed: the order is already shipped.",
                        tool_call_id=tool_call["id"]
                    )

                    messages.append(tool_message)


                elif order["amount"] > 500:

                    tool_message = ToolMessage(
                        content="Cancellation failed: orders above $500 cannot be cancelled.",
                        tool_call_id=tool_call["id"]
                    )

                    messages.append(tool_message)


                else:

                    # Use the trusted order_id we obtained not by llm from get_order()
                    tool_call["args"]["order_id"] = order["order_id"]

                    tool_message = cancel_order.invoke(tool_call)
                    messages.append(tool_message)

                    # Update local state using the ACTUAL result of cancel_order()
                    cancellation = json.loads(tool_message.content)

                    order["status"] = cancellation["status"]


            elif tool_call["name"] == "issue_refund":

                # Refund is only allowed after cancellation
                if order["status"] != "cancelled":

                    tool_message = ToolMessage(
                        content="Refund failed: the order must be successfully cancelled first.",
                        tool_call_id=tool_call["id"]
                    )

                    messages.append(tool_message)

                elif order.get("refunded", False):

                    tool_message = ToolMessage(
                        content="Refund failed: this order has already been refunded.",
                        tool_call_id=tool_call["id"]
                    )

                    messages.append(tool_message)

                else:

                    # Use trusted order ID
                    tool_call["args"]["order_id"] = order["order_id"]

                    tool_message = issue_refund.invoke(tool_call)
                    messages.append(tool_message)

                    refund = json.loads(tool_message.content)

                    order["refunded"] = True

 